# ✈️ Flight Delay Prediction — Spark ML Pipeline
> **Stack:** ClickHouse (JDBC) → Apache Spark → PySpark ML 
> **Target:** `arr_del15_label` (0 = Tepat Waktu, 1 = Delay ≥15 menit)  
> **Split Strategy:** Time-based (Train: 2024, Test: 2025)  
> **Models:** Logistic Regression · Random Forest · GBTClassifier  
> **Eksperimen Balancing:** 2 versi paralel —
> - **Versi A**: Class weighting saja (populasi penuh, tanpa undersampling)
> - **Versi B**: Hybrid — Random undersampling kelas mayoritas (ontime) ke rasio 70:30, lalu reweight berdasarkan rasio baru
>
> ⚠️ **Revisi penting:** `arr_hour` (dan turunannya `arr_hour_sin`/`arr_hour_cos`) dihapus dari fitur karena merupakan data leakage — jam kedatangan aktual hanya diketahui setelah pesawat landing, padahal target prediksi adalah delay kedatangan itu sendiri. `OriginAirportID`/`DestAirportID` (numerik mentah) juga diganti dengan `origin_index`/`dest_index` hasil `StringIndexer` karena ID lokasi bersifat kategorikal, bukan ordinal.


## 📦 1. IMPORT LIBRARY

In [1]:
import os
import sys
import time
import shutil
import numpy as np
import pandas as pd
import clickhouse_connect
import threading
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier,           # ← tambahan baru
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print('✅ Semua library berhasil di-import!')

✅ Semua library berhasil di-import!


## 🔥 2. INISIALISASI SPARK

In [2]:
# Bersihkan sesi lama jika masih nyangkut
if 'spark' in locals():
    try: spark.stop()
    except: pass

path_jar = os.path.join("..", "spark", "conf", "clickhouse-jdbc-0.6.3.jar")

if not os.path.exists(path_jar):
    print(f"❌ File JAR tidak ditemukan di '{path_jar}'!")
    print("Pastikan file ada di folder yang sama dengan notebook ini.")
else:
    print("✅ File JAR ditemukan! Memulai SparkSession...")

    python_path = sys.executable
    os.environ["PYSPARK_PYTHON"]        = python_path
    os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

    spark = SparkSession.builder \
        .appName("FlightDelayModelBuilder") \
        .config("spark.jars", path_jar) \
        .config("spark.driver.extraClassPath", path_jar) \
        .config("spark.executor.extraClassPath", path_jar) \
        .config("spark.executor.memory", "8g") \
        .config("spark.driver.memory", "8g") \
        .config("spark.driver.maxResultSize", "4g") \
        .config("spark.memory.offHeap.enabled", "true") \
        .config("spark.memory.offHeap.size", "1g") \
        .config("spark.memory.fraction", "0.7") \
        .config("spark.memory.storageFraction", "0.3") \
        .config("spark.pyspark.python", python_path) \
        .config("spark.pyspark.driver.python", python_path) \
        .config("spark.executorEnv.PYSPARK_PYTHON", python_path) \
        .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC -XX:G1HeapRegionSize=32m") \
        .config("spark.driver.extraJavaOptions", "-XX:+UseG1GC -XX:G1HeapRegionSize=32m") \
        .getOrCreate()

    print("✅ SparkSession berhasil terhubung!")
    print(f"   Spark version: {spark.version}")

✅ File JAR ditemukan! Memulai SparkSession...
✅ SparkSession berhasil terhubung!
   Spark version: 4.1.2


## 📥 3. MEMBACA DATA DARI CLICKHOUSE

In [3]:
# ── Konfigurasi koneksi ClickHouse ───────────────────────────────
CLICKHOUSE_URL = (
    "jdbc:clickhouse://13.215.79.3:8123/flight_delay"
    "?compress=false"
    "&connection_timeout=120000"
    "&socket_timeout=120000"
    "&http_connection_provider=HTTP_URL_CONNECTION"
)

print("📥 Menarik data dari ClickHouse (12 partisi bulan)...")
start_time = time.time()

df_all = spark.read.format("jdbc") \
    .option("url", CLICKHOUSE_URL) \
    .option("dbtable", "ontime_features") \
    .option("user", "default") \
    .option("password", "rahasia123") \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("partitionColumn", "flight_month") \
    .option("lowerBound", "1") \
    .option("upperBound", "6") \
    .option("numPartitions", "6") \
    .option("fetchsize", "5000") \
    .load()

print(f"✅ Schema dimuat dalam {time.time()-start_time:.1f} detik")
print(f"   Total kolom: {len(df_all.columns)}")

# ── Alternatif: baca dari Parquet lokal (uncomment jika perlu) ───
# df_all = spark.read.parquet("../data/ontime_features.parquet")

# Cek skema & ketersediaan kolom
print("📋 Skema kolom ontime_features:")
for col_name, dtype in df_all.dtypes:
    print(f"  {col_name}: {dtype}")

📥 Menarik data dari ClickHouse (12 partisi bulan)...
✅ Schema dimuat dalam 6.9 detik
   Total kolom: 44
📋 Skema kolom ontime_features:
  FlightDate: date
  IATA_CODE_Reporting_Airline: string
  Flight_Number_Reporting_Airline: string
  Origin: string
  Dest: string
  CRSDepTime: int
  flight_year: int
  flight_quarter: int
  flight_month: int
  flight_day: int
  day_of_week: int
  is_weekend: int
  season: string
  dep_hour: int
  arr_hour: int
  dep_time_bucket: string
  arr_time_bucket: string
  route: string
  same_state_route: int
  distance_bucket: int
  CRSElapsedTime: double
  Distance: double
  DistanceGroup: int
  OriginAirportID: int
  OriginState: string
  DestAirportID: int
  DestState: string
  route_avg_arr_delay_prev: double
  route_arr_delay_rate_prev: double
  route_cancel_rate_prev: double
  carrier_arr_delay_rate_prev: double
  carrier_cancel_rate_prev: double
  origin_arr_delay_rate_prev: double
  origin_cancel_rate_prev: double
  dest_arr_delay_rate_prev: double
  

## ✂️ 4. TIME-BASED SPLIT & CACHING

In [4]:
# # Definisikan pemisahan dasar langsung ke variabel target
# df_train_raw = df_all.filter(df_all.FlightDate < "2025-01-01")
# df_test_raw = df_all.filter(df_all.FlightDate >= "2025-01-01")

# # Data Training
# print("\n📥 Tahap A: Mengunduh Data Training (2021-2024) ke RAM...")
# start_train = time.time()

# for month in range(1, 13):
#     print(f"   ⏳ [Partisi {month}/12] Menyedot data penerbangan Bulan ke-{month:02d} dari AWS...")
#     # Ubah filter menggunakan 'flight_month'
#     df_train_raw.filter(df_train_raw.flight_month == month).cache().count()

# df_train_raw = df_train_raw.cache()
# train_count = df_train_raw.count()
# print(f"   ✔️ Sukses! {train_count:,} baris Data Training terkunci di RAM. Durasi: {time.time() - start_train:.2f} detik.")

# # Data Testing
# print("\n📥 Tahap B: Mengunduh Data Testing (Tahun 2025) ke RAM...")
# start_test = time.time()

# df_test_raw = df_test_raw.cache()
# test_count = df_test_raw.count()
# print(f"   ✔️ Sukses! {test_count:,} baris Data Testing terkunci di RAM. Durasi: {time.time() - start_test:.2f} detik.")


In [5]:
# client = clickhouse_connect.get_client(
#     host='13.215.79.3',
#     port=8123,
#     username='default',
#     password='rahasia123',
#     database='flight_delay',
# )

# def tarik_per_bulan(tahun_filter, label):
#     """Tarik data per bulan lalu gabungkan — hindari timeout koneksi."""
    
#     if tahun_filter == 'train':
#         bulan_list = [
#             (tahun, bulan)
#             for tahun in range(2023, 2025)   # 2023 s/d 2024
#             for bulan in range(1, 13)         # Januari s/d Desember
#         ]
#     else:
#         # 2025: tarik bulan yang tersedia saja
#         bulan_list = [(2025, m) for m in range(1, 13)]
    
#     chunks = []
#     for tahun, bulan in bulan_list:
#         print(f"   📦 [{label}] Bulan {tahun}-{bulan:02d}...", end=" ", flush=True)
#         t0 = time.time()
#         try:
#             df_chunk = client.query_df(f"""
#                 SELECT * FROM ontime_features
#                 WHERE flight_year = {tahun}
#                   AND flight_month = {bulan}
#             """)
#             if len(df_chunk) > 0:
#                 chunks.append(df_chunk)
#                 print(f"✅ {len(df_chunk):,} baris ({time.time()-t0:.1f}s)")
#             else:
#                 print(f"⚠️ kosong, skip")
#         except Exception as e:
#             print(f"❌ Gagal: {e}")
#             continue  # skip bulan ini, lanjut ke berikutnya
    
#     if not chunks:
#         raise ValueError(f"Tidak ada data berhasil ditarik untuk {label}")
    
#     result = pd.concat(chunks, ignore_index=True)
#     return result


# # ── Tarik Training (2023-2024) ─────────────────────────────────────────
# print("📥 Menarik data Training (2023-2024) per bulan...")
# t0 = time.time()
# df_train_pd = tarik_per_bulan('train', 'Train')
# print(f"✅ Training selesai! {len(df_train_pd):,} baris — {time.time()-t0:.1f} detik\n")

# # ── Tarik Testing (2025) ──────────────────────────────────────────
# print("📥 Menarik data Testing (2025) per bulan...")
# t0 = time.time()
# df_test_pd = tarik_per_bulan('test', 'Test')
# print(f"✅ Testing selesai! {len(df_test_pd):,} baris — {time.time()-t0:.1f} detik\n")



In [6]:
# # ── STEP 1: Simpan hasil tarik ke Parquet lokal ──────────────────
# print("💾 Menyimpan ke Parquet lokal...")

# os.makedirs("../data", exist_ok=True)

# df_train_pd.to_parquet("../data/train_2024.parquet", index=False)
# print(f"   ✅ Training tersimpan: {len(df_train_pd):,} baris")
# del df_train_pd  # bebaskan RAM Pandas

# df_test_pd.to_parquet("../data/test_2025.parquet", index=False)
# print(f"   ✅ Testing tersimpan: {len(df_test_pd):,} baris")
# del df_test_pd   # bebaskan RAM Pandas

# print("✅ Semua data tersimpan ke Parquet!")

In [7]:
# ── STEP 2: Load Parquet ke Spark (RAM Pandas sudah kosong) ──────
print("⚡ Load Parquet → Spark DataFrame...")

df_train_raw = spark.read.parquet("../data/train_2024.parquet").cache()
train_count  = df_train_raw.count()
print(f"✅ Training terkunci di RAM: {train_count:,} baris")

df_test_raw = spark.read.parquet("../data/test_2025.parquet").cache()
test_count  = df_test_raw.count()
print(f"✅ Testing terkunci di RAM: {test_count:,} baris")

print(f"\n🎉 Semua data siap!")
print(f"   Training : {train_count:,} baris")
print(f"   Testing  : {test_count:,} baris")

⚡ Load Parquet → Spark DataFrame...
✅ Training terkunci di RAM: 11,728,023 baris
✅ Testing terkunci di RAM: 4,756,367 baris

🎉 Semua data siap!
   Training : 11,728,023 baris
   Testing  : 4,756,367 baris


## 🧹 5. CLEANING & VALIDASI LABEL

In [8]:
# Jalankan ini dulu untuk tahu nilai apa saja yang ada
df_train_raw.select("arr_del15_label").distinct().show(50)

+---------------+
|arr_del15_label|
+---------------+
|              1|
|              0|
+---------------+



In [9]:
print("🧹 Membersihkan label arr_del15_label...")

def clean_label(df):
    return (
        df
        # Paksa kolom jadi string dulu — tangani tipe apapun
        .withColumn("arr_del15_label", F.col("arr_del15_label").cast("string"))
        # Ganti semua varian \N dan string kosong jadi null
        .withColumn(
            "arr_del15_label",
            F.when(
                F.trim(F.col("arr_del15_label")).isin(["\\N", "\\\\N", "", "NULL", "null", "nan"]),
                F.lit(None)
            ).otherwise(F.trim(F.col("arr_del15_label")))
        )
        # Hapus null
        .filter(F.col("arr_del15_label").isNotNull())
        # Cast dua tahap: string → double → int (aman untuk "0.0" dan "1.0")
        .withColumn(
            "arr_del15_label",
            F.col("arr_del15_label").cast("double").cast("int")
        )
        # Hanya label 0 dan 1
        .filter(F.col("arr_del15_label").isin([0, 1]))
    )

df_train_raw = clean_label(df_train_raw)
df_test_raw  = clean_label(df_test_raw)

# Cek isi label tanpa operasi yang berisiko
train_clean = df_train_raw.count()
test_clean  = df_test_raw.count()

print(f"   Training bersih : {train_clean:,} baris")
print(f"   Testing bersih  : {test_clean:,} baris")

print("\n📊 Distribusi label Training:")
df_train_raw.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

print("📊 Distribusi label Testing:")
df_test_raw.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

🧹 Membersihkan label arr_del15_label...
   Training bersih : 11,728,023 baris
   Testing bersih  : 4,756,367 baris

📊 Distribusi label Training:
+---------------+-------+
|arr_del15_label|  count|
+---------------+-------+
|              0|9307806|
|              1|2420217|
+---------------+-------+

📊 Distribusi label Testing:
+---------------+-------+
|arr_del15_label|  count|
+---------------+-------+
|              0|3719567|
|              1|1036800|
+---------------+-------+



## ⚖️ 6. HANDLE IMBALANCE — 2 VERSI BALANCING (WEIGHTING vs HYBRID)

In [ ]:
# ── Distribusi kelas asli (sebelum balancing apapun) ──────────────
count_total  = df_train_raw.count()
count_delay  = df_train_raw.filter(F.col("arr_del15_label") == 1).count()
count_ontime = df_train_raw.filter(F.col("arr_del15_label") == 0).count()

print(f"⚖️  Distribusi asli training:")
print(f"   Ontime (0) : {count_ontime:,}")
print(f"   Delay  (1) : {count_delay:,}")
print(f"   Imbalance ratio : {count_ontime/count_delay:.2f}:1")

# ── time_weight dipakai di kedua versi (turunkan bobot data lama) ──
def add_time_weight(df):
    return df.withColumn(
        "time_weight",
        F.when(F.col("FlightDate") < "2023-01-01", 0.5).otherwise(1.0)
    )

# ════════════════════════════════════════════════════════════════
# VERSI A — CLASS WEIGHTING SAJA (tanpa undersampling)
# ════════════════════════════════════════════════════════════════
weight_delay_A  = count_total / (2.0 * count_delay)
weight_ontime_A = count_total / (2.0 * count_ontime)

print(f"\n📌 VERSI A — Weighting saja (rasio asli {count_ontime/count_delay:.2f}:1)")
print(f"   weight_delay  : {weight_delay_A:.4f}")
print(f"   weight_ontime : {weight_ontime_A:.4f}")

df_train_A = add_time_weight(df_train_raw) \
    .withColumn(
        "class_weight",
        F.when(F.col("arr_del15_label") == 1, weight_delay_A).otherwise(weight_ontime_A)
    ) \
    .withColumn("final_weight", F.col("class_weight") * F.col("time_weight"))

# ════════════════════════════════════════════════════════════════
# VERSI B — HYBRID: RANDOM UNDERSAMPLING (→ 70:30) + REWEIGHTING
# ════════════════════════════════════════════════════════════════
# Target: ontime:delay = 70:30 setelah undersampling kelas mayoritas.
# Hitung berapa banyak baris ontime yang perlu DIPERTAHANKAN agar
# rasio ontime:delay menjadi tepat 70:30 (delay TIDAK disentuh).
TARGET_ONTIME_RATIO = 0.70
TARGET_DELAY_RATIO  = 0.30

# ontime_target / delay_count = TARGET_ONTIME_RATIO / TARGET_DELAY_RATIO
ontime_target_count = int(count_delay * (TARGET_ONTIME_RATIO / TARGET_DELAY_RATIO))
ontime_target_count = min(ontime_target_count, count_ontime)  # safety guard

fraction_keep_ontime = ontime_target_count / count_ontime

print(f"\n📌 VERSI B — Hybrid: Undersampling → 70:30, lalu reweight")
print(f"   Ontime tersedia        : {count_ontime:,}")
print(f"   Ontime target (70%)    : {ontime_target_count:,}")
print(f"   Delay dipertahankan utuh: {count_delay:,} (100%, tidak diundersample)")
print(f"   Fraction sample ontime  : {fraction_keep_ontime:.4f}")

df_delay_full     = df_train_raw.filter(F.col("arr_del15_label") == 1)
df_ontime_sampled = df_train_raw.filter(F.col("arr_del15_label") == 0) \
                                  .sample(withReplacement=False,
                                          fraction=fraction_keep_ontime,
                                          seed=42)

df_train_hybrid_raw = df_delay_full.unionByName(df_ontime_sampled)

# Hitung ulang rasio AKTUAL setelah sampling (sample() bersifat probabilistik,
# jadi count aktual bisa sedikit berbeda dari target — ini yang dipakai utk weight)
count_total_B  = df_train_hybrid_raw.count()
count_delay_B  = df_delay_full.count()
count_ontime_B = df_train_hybrid_raw.count() - count_delay_B

print(f"\n   Hasil aktual setelah undersampling:")
print(f"   Ontime (0) : {count_ontime_B:,} ({count_ontime_B/count_total_B*100:.1f}%)")
print(f"   Delay  (1) : {count_delay_B:,} ({count_delay_B/count_total_B*100:.1f}%)")
print(f"   Rasio baru : {count_ontime_B/count_delay_B:.2f}:1")

# ── Reweight berdasarkan RASIO BARU (bukan rasio asli!) ───────────
# Ini kunci dari hybrid yang benar: weight dihitung dari distribusi
# yang SUDAH di-undersample, bukan dari distribusi populasi asli.
# Kalau pakai weight dari rasio asli di sini, akan terjadi
# double-correction yang membuat model bias berlebihan ke kelas delay.
weight_delay_B  = count_total_B / (2.0 * count_delay_B)
weight_ontime_B = count_total_B / (2.0 * count_ontime_B)

print(f"\n   weight_delay  (rasio baru) : {weight_delay_B:.4f}")
print(f"   weight_ontime (rasio baru) : {weight_ontime_B:.4f}")

df_train_B = add_time_weight(df_train_hybrid_raw) \
    .withColumn(
        "class_weight",
        F.when(F.col("arr_del15_label") == 1, weight_delay_B).otherwise(weight_ontime_B)
    ) \
    .withColumn("final_weight", F.col("class_weight") * F.col("time_weight"))

print("\n✅ Dua versi training set siap: df_train_A (weighting) & df_train_B (hybrid undersample+reweight)")


## ⚙️ 7. FEATURE ENGINEERING & PREPROCESSING

In [ ]:
print("⚙️ Memulai transformasi fitur...")

# ── StringIndexer untuk kategorikal ──────────────────────────────
# Catatan: di-fit pada df_train_A (versi weighting, populasi penuh)
# supaya vocabulary kategori selalu mencakup data terlengkap.
# Origin & Dest ditambahkan sebagai categorical index yang benar —
# sebelumnya OriginAirportID/DestAirportID dipakai sebagai NUMERIK
# MENTAH di FEATURE_COLS, padahal itu hanya label ID, bukan besaran
# ordinal. Ini bisa menyesatkan model (terutama Logistic Regression).
indexer_season  = StringIndexer(inputCol="season",                      outputCol="season_index",  handleInvalid="keep")
indexer_airline = StringIndexer(inputCol="IATA_CODE_Reporting_Airline", outputCol="airline_index", handleInvalid="keep")
indexer_origin  = StringIndexer(inputCol="Origin",                      outputCol="origin_index",  handleInvalid="keep")
indexer_dest    = StringIndexer(inputCol="Dest",                        outputCol="dest_index",    handleInvalid="keep")

fit_season  = indexer_season.fit(df_train_A)
fit_airline = indexer_airline.fit(df_train_A)
fit_origin  = indexer_origin.fit(df_train_A)
fit_dest    = indexer_dest.fit(df_train_A)

def apply_indexers(df):
    df = fit_season.transform(df)
    df = fit_airline.transform(df)
    df = fit_origin.transform(df)
    df = fit_dest.transform(df)
    return df

# ── Terapkan ke KEDUA versi training + test ───────────────────────
df_train_A_indexed = apply_indexers(df_train_A)
df_train_B_indexed = apply_indexers(df_train_B)
df_test_indexed     = apply_indexers(df_test_raw)

# ── PENTING: lepas metadata kategorikal dari hasil StringIndexer ──
# StringIndexer menempelkan metadata ml_attr (nominal, num_values=N)
# ke kolom output. VectorAssembler mewarisi metadata ini ke vector
# "features", dan RandomForest/GBT di Spark MLlib MEMBACA metadata
# tsb untuk otomatis mendeteksi fitur kategorikal. Origin/Dest punya
# ratusan nilai unik (cardinality tinggi) — jauh melebihi maxBins
# default (32) — sehingga tree-based model gagal training dengan
# error "categorical feature X has N values, maxBins too small".
#
# Solusi: treat origin_index/dest_index/season_index/airline_index
# sebagai NUMERIK biasa (ordinal), bukan kategorikal murni, dengan
# cara "membersihkan" metadata via cast double -> string -> double.
# Trik ini membuang ml_attr karena kolom baru dianggap hasil operasi
# baru, bukan output StringIndexer secara langsung.
def strip_categorical_metadata(df, cols):
    for c in cols:
        df = df.withColumn(c, F.col(c).cast("string").cast("double"))
    return df

INDEX_COLS_TO_STRIP = ["season_index", "airline_index", "origin_index", "dest_index"]
df_train_A_indexed = strip_categorical_metadata(df_train_A_indexed, INDEX_COLS_TO_STRIP)
df_train_B_indexed = strip_categorical_metadata(df_train_B_indexed, INDEX_COLS_TO_STRIP)
df_test_indexed     = strip_categorical_metadata(df_test_indexed,    INDEX_COLS_TO_STRIP)

# ── Cast kolom ke double ──────────────────────────────────────────
# arr_hour DIHAPUS dari daftar ini — lihat penjelasan leakage di bawah.
KOLOM_NUMERIK = [
    "route_avg_arr_delay_prev", "carrier_arr_delay_rate_prev",
    "flight_month", "dep_hour",
    "is_weekend", "same_state_route",
    "day_of_week", "CRSElapsedTime", "Distance", "flight_day",
    "route_arr_delay_rate_prev", "route_carrier_arr_delay_rate_prev",
    "route_cancel_rate_prev",
    "origin_arr_delay_rate_prev", "origin_cancel_rate_prev",
    "dest_arr_delay_rate_prev",   "dest_cancel_rate_prev",
    "carrier_cancel_rate_prev",   "distance_bucket",
]

def cast_numeric(df):
    for col in KOLOM_NUMERIK:
        df = df.withColumn(col, F.col(col).cast("double"))
    return df

df_train_A_indexed = cast_numeric(df_train_A_indexed)
df_train_B_indexed = cast_numeric(df_train_B_indexed)
df_test_indexed     = cast_numeric(df_test_indexed)

# ── Flag has_route_history (sebelum fillna) ───────────────────────
def add_history_flag(df):
    return df.withColumn(
        "has_route_history",
        F.when(F.col("route_avg_arr_delay_prev").isNull(), 0.0).otherwise(1.0)
    )

df_train_A_indexed = add_history_flag(df_train_A_indexed)
df_train_B_indexed = add_history_flag(df_train_B_indexed)
df_test_indexed     = add_history_flag(df_test_indexed)

# ── Imputasi median (dihitung dari df_train_A = populasi penuh) ───
# Median dihitung sekali dari training penuh (versi A) agar konsisten
# dipakai untuk versi A, versi B, maupun test — menghindari kebocoran
# informasi test ke proses imputasi, sekaligus menjaga skala nilai
# yang konsisten antar dua eksperimen balancing.
KOLOM_MEDIAN = [
    "route_avg_arr_delay_prev", "carrier_arr_delay_rate_prev",
    "route_arr_delay_rate_prev", "route_carrier_arr_delay_rate_prev",
    "route_cancel_rate_prev",
    "origin_arr_delay_rate_prev", "origin_cancel_rate_prev",
    "dest_arr_delay_rate_prev", "dest_cancel_rate_prev",
    "carrier_cancel_rate_prev",
]
medians = {}
for col in KOLOM_MEDIAN:
    medians[col] = df_train_A_indexed.approxQuantile(col, [0.5], 0.01)[0]
    print(f"  Median {col}: {medians[col]:.4f}")

def impute_median(df):
    for col in KOLOM_MEDIAN:
        df = df.fillna(medians[col], subset=[col])
    return df

df_train_A_indexed = impute_median(df_train_A_indexed)
df_train_B_indexed = impute_median(df_train_B_indexed)
df_test_indexed     = impute_median(df_test_indexed)

OTHER_COLS = [
    "flight_month", "dep_hour", "is_weekend",
    "same_state_route", "day_of_week", "CRSElapsedTime",
    "Distance", "flight_day", "distance_bucket",
]

def fillna_other(df):
    return df.fillna(0.0, subset=OTHER_COLS)

df_train_A_indexed = fillna_other(df_train_A_indexed)
df_train_B_indexed = fillna_other(df_train_B_indexed)
df_test_indexed     = fillna_other(df_test_indexed)
print("\n✅ Imputasi selesai!")

# ── Encoding siklikal ─────────────────────────────────────────────
# ⚠️ arr_hour_sin/cos DIHAPUS — arr_hour adalah jam KEDATANGAN AKTUAL
# yang hanya diketahui SETELAH pesawat landing. Karena target kita
# arr_del15_label (delay kedatangan), memasukkan arr_hour sebagai
# fitur = data leakage: model "menyontek" dari akibat delay itu
# sendiri, bukan memprediksi dari informasi yang tersedia sebelum
# penerbangan terjadi. dep_hour aman dipakai karena itu jadwal CRS
# yang sudah diketahui sebelum boarding.
def add_cyclical_features(df):
    return df \
        .withColumn("dep_hour_sin",  F.sin(2 * 3.14159 * F.col("dep_hour")     / 24)) \
        .withColumn("dep_hour_cos",  F.cos(2 * 3.14159 * F.col("dep_hour")     / 24)) \
        .withColumn("month_sin",     F.sin(2 * 3.14159 * F.col("flight_month") / 12)) \
        .withColumn("month_cos",     F.cos(2 * 3.14159 * F.col("flight_month") / 12)) \
        .withColumn("dow_sin",       F.sin(2 * 3.14159 * F.col("day_of_week")  / 7)) \
        .withColumn("dow_cos",       F.cos(2 * 3.14159 * F.col("day_of_week")  / 7))

df_train_A_indexed = add_cyclical_features(df_train_A_indexed)
df_train_B_indexed = add_cyclical_features(df_train_B_indexed)
df_test_indexed     = add_cyclical_features(df_test_indexed)

# ── Fitur interaksi ───────────────────────────────────────────────
def add_interaction_features(df):
    return df \
        .withColumn(
            "route_x_carrier_risk",
            F.col("route_arr_delay_rate_prev") * F.col("carrier_arr_delay_rate_prev")
        ) \
        .withColumn(
            "origin_x_dest_risk",
            F.col("origin_arr_delay_rate_prev") * F.col("dest_arr_delay_rate_prev")
        ) \
        .withColumn(
            "carrier_rate_x_peak_hour",
            F.col("carrier_arr_delay_rate_prev") * F.col("dep_hour_sin")
        )

df_train_A_indexed = add_interaction_features(df_train_A_indexed)
df_train_B_indexed = add_interaction_features(df_train_B_indexed)
df_test_indexed     = add_interaction_features(df_test_indexed)

print("✅ Feature engineering selesai untuk Versi A & Versi B!")


In [ ]:
# ── Definisi fitur final & VectorAssembler ────────────────────────
FEATURE_COLS = [
    # Temporal (siklikal) — arr_hour_sin/cos DIHAPUS (leakage, lihat sel sebelumnya)
    "dep_hour_sin", "dep_hour_cos",
    "month_sin",    "month_cos",
    "dow_sin",      "dow_cos",
    "flight_day",

    # Rute & maskapai — OriginAirportID/DestAirportID (numerik mentah)
    # DIGANTI dengan origin_index/dest_index (hasil StringIndexer)
    "airline_index", "origin_index", "dest_index",
    "season_index",

    # Operasional
    "Distance", "CRSElapsedTime", "distance_bucket",
    "has_route_history",

    # Historis (paling penting!) — route_cancel_rate_prev DITAMBAHKAN
    "route_avg_arr_delay_prev",
    "route_arr_delay_rate_prev",
    "route_cancel_rate_prev",
    "route_carrier_arr_delay_rate_prev",
    "carrier_arr_delay_rate_prev",
    "carrier_cancel_rate_prev",
    "origin_arr_delay_rate_prev",
    "origin_cancel_rate_prev",
    "dest_arr_delay_rate_prev",
    "dest_cancel_rate_prev",

    # Interaksi
    "route_x_carrier_risk",
    "origin_x_dest_risk",
    "carrier_rate_x_peak_hour",
]

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features", handleInvalid="keep")

# ── VERSI A (weighting saja) ───────────────────────────────────────
df_train_final_A = assembler.transform(df_train_A_indexed) \
    .select("features", "arr_del15_label", "final_weight").cache()

# ── VERSI B (hybrid undersampling 70:30 + reweight) ────────────────
df_train_final_B = assembler.transform(df_train_B_indexed) \
    .select("features", "arr_del15_label", "final_weight").cache()

# ── Test set (sama untuk kedua versi, tidak pernah disampling) ─────
df_test_final = assembler.transform(df_test_indexed) \
    .select("features", "arr_del15_label").cache()

print(f"✅ VectorAssembler selesai! Total fitur: {len(FEATURE_COLS)}")
print(f"\n   [Versi A] df_train_final_A sample:"); df_train_final_A.limit(3).show()
print(f"   [Versi B] df_train_final_B sample:"); df_train_final_B.limit(3).show()
print(f"   df_test_final sample:");              df_test_final.limit(3).show()


## 🔍 7.1. DIAGNOSTIK DATA

In [ ]:
print("🔍 Diagnostik sebelum training...")

print("\n📊 Distribusi label Training ASLI (sebelum balancing):")
df_train_raw.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

print("📊 [Versi A] Distribusi label Training (weighting saja, populasi penuh):")
df_train_A_indexed.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

print("📊 [Versi B] Distribusi label Training (hybrid undersample 70:30):")
df_train_B_indexed.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

print("📊 Distribusi label Testing:")
df_test_raw.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

pos_test = df_test_raw.filter(F.col("arr_del15_label") == 1).count()
neg_test = df_test_raw.filter(F.col("arr_del15_label") == 0).count()
baseline = max(pos_test, neg_test) / (pos_test + neg_test)
print(f"🔢 Positive rate test : {pos_test/(pos_test+neg_test):.4f}")
print(f"🔢 Baseline accuracy  : {baseline:.4f}")

print("\n🧪 Null count pada fitur kunci (training Versi A):")
null_check_cols = [
    'route_avg_arr_delay_prev', 'carrier_arr_delay_rate_prev',
    'dep_hour', 'is_weekend', 'season_index', 'distance_bucket',
    'origin_index', 'dest_index',
]
df_train_A_indexed.select(
    [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in null_check_cols]
).show()

# Sanity check: pastikan arr_hour TIDAK ada di FEATURE_COLS (cek leakage)
leak_check = [c for c in FEATURE_COLS if 'arr_hour' in c]
if leak_check:
    print(f"\n🚨 PERINGATAN: ditemukan fitur leakage di FEATURE_COLS: {leak_check}")
else:
    print("\n✅ Sanity check leakage: tidak ada fitur arr_hour di FEATURE_COLS.")


## 🤖 8. TRAINING MODEL

In [ ]:
print("🤖 Menginisialisasi & melatih semua model untuk KEDUA versi balancing...")

# ── Dataset training per versi balancing ──────────────────────────
TRAINING_VERSIONS = {
    "A_Weighting":       df_train_final_A,
    "B_Hybrid_Undersample": df_train_final_B,
}

# Catatan maxBins: season_index/airline_index/origin_index/dest_index
# sudah dibersihkan metadatanya di sel feature engineering (di-treat
# sebagai numerik/ordinal biasa, bukan kategorikal murni). Karena itu,
# RF & GBT TIDAK akan mendeteksinya sebagai categorical feature lagi,
# sehingga maxBins default (32) sudah cukup. maxBins tetap dinaikkan
# sedikit ke 64 di sini sebagai pengaman tambahan untuk split numerik
# yang lebih presisi pada fitur rate-based (granularitas float).
def build_models():
    """Re-instantiate model baru setiap kali — model Spark ML tidak boleh
    di-fit ulang pada estimator yang sama tanpa membuat instance baru."""
    return {
        "Logistic Regression": LogisticRegression(
            featuresCol="features",
            labelCol="arr_del15_label",
            weightCol="final_weight",
            maxIter=100,
        ),
        "Random Forest": RandomForestClassifier(
            featuresCol="features",
            labelCol="arr_del15_label",
            weightCol="final_weight",
            numTrees=50,
            maxDepth=8,
            maxBins=64,
            seed=42,
        ),
        "GBT Classifier": GBTClassifier(
            featuresCol="features",
            labelCol="arr_del15_label",
            weightCol="final_weight",  # ← GBT Spark MLlib mendukung weightCol
            maxIter=100,
            maxDepth=5,
            maxBins=64,
            stepSize=0.05,             # learning rate
            subsamplingRate=0.8,
            seed=42,
        ),
    }

# trained_spark_models kini berstruktur nested:
#   trained_spark_models[versi][nama_model] = model_terlatih
trained_spark_models = {}

for versi_name, df_train_set in TRAINING_VERSIONS.items():
    print(f"\n{'='*70}")
    print(f"🚀 Training untuk VERSI: {versi_name}")
    print(f"{'='*70}")

    models = build_models()
    trained_spark_models[versi_name] = {}

    for name, model in models.items():
        print(f"   ⏳ Training {name} [{versi_name}]...")
        t0 = time.time()
        try:
            trained_spark_models[versi_name][name] = model.fit(df_train_set)
            print(f"   ✅ {name} selesai dalam {time.time()-t0:.1f} detik")
        except Exception as e:
            print(f"   ❌ {name} gagal: {e}")
            trained_spark_models[versi_name][name] = None

print("\n🎉 Semua model (2 versi × 3 algoritma) selesai di-training!")


## 🔎 8.1. INSPEKSI MODEL (FEATURE IMPORTANCE)

In [ ]:
print("🔎 Inspeksi koefisien & feature importance untuk setiap versi...")

for versi_name, models_dict in trained_spark_models.items():
    print(f"\n{'#'*70}")
    print(f"# VERSI: {versi_name}")
    print(f"{'#'*70}")

    # Logistic Regression coefficients
    lr_model = models_dict.get("Logistic Regression")
    if lr_model:
        print(f"\n📈 [{versi_name}] Logistic Regression — Koefisien (Top 15 |coef|):")
        coeffs = lr_model.coefficients.toArray()
        for name, coef in sorted(zip(FEATURE_COLS, coeffs), key=lambda x: abs(x[1]), reverse=True)[:15]:
            print(f"  {name:40s}: {coef:+.6f}")

    # Random Forest feature importances
    rf_model = models_dict.get("Random Forest")
    if rf_model:
        print(f"\n🌲 [{versi_name}] Random Forest — Feature Importance (Top 15):")
        importances = rf_model.featureImportances.toArray()
        for name, imp in sorted(zip(FEATURE_COLS, importances), key=lambda x: x[1], reverse=True)[:15]:
            print(f"  {name:40s}: {imp:.6f}")

    # GBT feature importances
    gbt_model = models_dict.get("GBT Classifier")
    if gbt_model:
        print(f"\n🌳 [{versi_name}] GBT Classifier — Feature Importance (Top 15):")
        importances = gbt_model.featureImportances.toArray()
        for name, imp in sorted(zip(FEATURE_COLS, importances), key=lambda x: x[1], reverse=True)[:15]:
            print(f"  {name:40s}: {imp:.6f}")


## 📊 9. EVALUASI MODEL

In [ ]:
print("🕵️ Evaluasi + Threshold Tuning — membandingkan Versi A vs Versi B...")

evaluator_roc = BinaryClassificationEvaluator(
    labelCol="arr_del15_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
evaluator_pr = BinaryClassificationEvaluator(
    labelCol="arr_del15_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
)

# UDF ekstrak probabilitas kelas positif (didefinisikan sekali di luar loop)
extract_prob = F.udf(lambda v: float(v[1]), "double")

final_eval_rows = []
threshold_rows  = []

for versi_name, models_dict in trained_spark_models.items():
    for name, model in models_dict.items():
        if model is None:
            print(f"\n⚠️ Skip {name} [{versi_name}] — model tidak tersedia")
            continue

        label_tampil = f"{name} ({versi_name})"
        print(f"\n⏳ Memproses: {label_tampil}...")

        # ── Cache predictions (selalu pakai df_test_final yang SAMA,
        #    test set tidak pernah disampling, jadi perbandingan adil) ──
        predictions = model.transform(df_test_final).cache()

        # ── 1. ROC-AUC & PR-AUC (pakai Spark evaluator) ─────────────
        roc_auc = evaluator_roc.evaluate(predictions)
        pr_auc  = evaluator_pr.evaluate(predictions)

        # ── 2. Confusion Matrix (pakai Spark count) ──────────────────
        pred_labels = predictions.select("arr_del15_label", "prediction")
        cm = pred_labels.groupBy("arr_del15_label", "prediction").count().collect()

        cm_dict = {(int(row["arr_del15_label"]), int(row["prediction"])): row["count"] for row in cm}
        tp = cm_dict.get((1, 1), 0)
        tn = cm_dict.get((0, 0), 0)
        fp = cm_dict.get((0, 1), 0)
        fn = cm_dict.get((1, 0), 0)

        total       = tp + tn + fp + fn
        accuracy    = (tp + tn) / total if total > 0 else 0.0
        precision   = tp / (tp + fp)    if (tp + fp) > 0 else 0.0
        recall      = tp / (tp + fn)    if (tp + fn) > 0 else 0.0
        f1          = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        specificity = tn / (tn + fp)    if (tn + fp) > 0 else 0.0

        print(f"   Confusion Matrix → TP={tp:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}")
        print(f"   Accuracy={accuracy:.4f} | Precision={precision:.4f} | Recall={recall:.4f}")
        print(f"   F1={f1:.4f} | ROC-AUC={roc_auc:.4f} | PR-AUC={pr_auc:.4f} | Specificity={specificity:.4f}")

        final_eval_rows.append({
            "Versi Balancing": versi_name,
            "Algoritma"      : name,
            "Accuracy"       : f"{accuracy:.4f}",
            "Precision"      : f"{precision:.4f}",
            "Recall"         : f"{recall:.4f}",
            "Specificity"    : f"{specificity:.4f}",
            "F1-Score"       : f"{f1:.4f}",
            "ROC-AUC"        : f"{roc_auc:.4f}",
            "PR-AUC"         : f"{pr_auc:.4f}",
        })

        # ── 3. Threshold Tuning (toPandas SEKALI, langsung di sini) ──
        df_local = predictions.select(
            "arr_del15_label",
            extract_prob(F.col("probability")).alias("prob_delay")
        ).sample(fraction=0.2, seed=42).toPandas()

        y_true  = df_local["arr_del15_label"].values
        y_proba = df_local["prob_delay"].values

        best_f1, best_thresh, best_prec, best_rec = 0, 0.5, 0, 0
        for thresh in np.arange(0.10, 0.90, 0.01):
            y_pred = (y_proba >= thresh).astype(int)
            tp_t = ((y_pred == 1) & (y_true == 1)).sum()
            fp_t = ((y_pred == 1) & (y_true == 0)).sum()
            fn_t = ((y_pred == 0) & (y_true == 1)).sum()
            prec_t = tp_t / (tp_t + fp_t + 1e-9)
            rec_t  = tp_t / (tp_t + fn_t + 1e-9)
            f1_t   = 2 * prec_t * rec_t / (prec_t + rec_t + 1e-9)
            if f1_t > best_f1:
                best_f1, best_thresh, best_prec, best_rec = f1_t, thresh, prec_t, rec_t

        print(f"   🎯 Threshold optimal: {best_thresh:.2f} → F1={best_f1:.4f} | Precision={best_prec:.4f} | Recall={best_rec:.4f}")

        threshold_rows.append({
            "Versi Balancing": versi_name,
            "Algoritma"      : name,
            "Threshold"      : f"{best_thresh:.2f}",
            "F1"             : f"{best_f1:.4f}",
            "Precision"      : f"{best_prec:.4f}",
            "Recall"         : f"{best_rec:.4f}",
        })

        # ── ✅ Unpersist SEGERA setelah semua pemakaian selesai ───────
        predictions.unpersist()
        del df_local, y_true, y_proba

# ── Tabel ringkasan akhir: bandingkan Versi A vs Versi B ───────────
print("\n" + "=" * 110)
print("📊 TABEL EVALUASI AKHIR — PERBANDINGAN VERSI A (Weighting) vs VERSI B (Hybrid Undersample+Reweight)")
print("=" * 110)
df_final_eval = pd.DataFrame(final_eval_rows)
print(df_final_eval.to_string(index=False))
print("=" * 110)

print("\n" + "=" * 90)
print("📊 RINGKASAN THRESHOLD TUNING (per versi & algoritma)")
print("=" * 90)
df_threshold = pd.DataFrame(threshold_rows)
print(df_threshold.to_string(index=False))

baseline_acc = neg_test / (pos_test + neg_test)
print(f"\n📌 Baseline majority-class accuracy: {baseline_acc:.4f}")

# ── Pivot ringkas untuk memudahkan perbandingan langsung di laporan ─
print("\n" + "=" * 90)
print("📊 PERBANDINGAN F1-SCORE SIDE-BY-SIDE (Versi A vs Versi B)")
print("=" * 90)
pivot_f1 = df_final_eval.pivot(index="Algoritma", columns="Versi Balancing", values="F1-Score")
print(pivot_f1.to_string())

print("\n" + "=" * 90)
print("📊 PERBANDINGAN ROC-AUC SIDE-BY-SIDE (Versi A vs Versi B)")
print("=" * 90)
pivot_roc = df_final_eval.pivot(index="Algoritma", columns="Versi Balancing", values="ROC-AUC")
print(pivot_roc.to_string())


## 🎯 10. THRESHOLD TUNING (OPTIMAL F1)

In [17]:
# print("🎯 Mencari threshold optimal (maksimalkan F1-Score)...")

# # UDF untuk ekstrak probabilitas kelas positif
# extract_prob = F.udf(lambda v: float(v[1]), "double")

# threshold_results = []

# for name, predictions in all_predictions.items():
#     print(f"\n🔍 Model: {name}")

#     # Tarik ke Pandas sekali — jauh lebih efisien dari 60x Spark jobs
#     df_local = predictions.select(
#         "arr_del15_label",
#         extract_prob(F.col("probability")).alias("prob_delay")
#     ).toPandas()

#     y_true  = df_local["arr_del15_label"].values
#     y_proba = df_local["prob_delay"].values

#     best_f1, best_thresh, best_prec, best_rec = 0, 0.5, 0, 0

#     for thresh in np.arange(0.10, 0.90, 0.01):
#         y_pred = (y_proba >= thresh).astype(int)
#         tp = ((y_pred == 1) & (y_true == 1)).sum()
#         fp = ((y_pred == 1) & (y_true == 0)).sum()
#         fn = ((y_pred == 0) & (y_true == 1)).sum()
#         prec = tp / (tp + fp + 1e-9)
#         rec  = tp / (tp + fn + 1e-9)
#         f1   = 2 * prec * rec / (prec + rec + 1e-9)
#         if f1 > best_f1:
#             best_f1, best_thresh, best_prec, best_rec = f1, thresh, prec, rec

#     threshold_results.append({
#         "Algoritma"          : name,
#         "Optimal Threshold"  : f"{best_thresh:.2f}",
#         "F1 @ Optimal"       : f"{best_f1:.4f}",
#         "Precision @ Optimal": f"{best_prec:.4f}",
#         "Recall @ Optimal"   : f"{best_rec:.4f}",
#     })
#     print(f"   Threshold optimal: {best_thresh:.2f}")
#     print(f"   F1={best_f1:.4f} | Precision={best_prec:.4f} | Recall={best_rec:.4f}")

# print("\n" + "=" * 75)
# print("📊 RINGKASAN THRESHOLD TUNING")
# print("=" * 75)
# print(pd.DataFrame(threshold_results).to_string(index=False))

## 💾 11. SIMPAN MODEL & ENCODER

In [ ]:
# os.makedirs("../models", exist_ok=True)

# # ── Simpan semua model (struktur nested: versi -> nama_model) ─────
# for versi_name, models_dict in trained_spark_models.items():
#     for name, model in models_dict.items():
#         if model is None:
#             print(f"⚠️ Skip {name} [{versi_name}] — model tidak tersedia")
#             continue

#         folder_name = f"{versi_name}__{name.lower().replace(' ', '_')}"
#         save_path = os.path.join("..", "models", folder_name)
#         if os.path.exists(save_path):
#             shutil.rmtree(save_path)
#         model.save(save_path)
#         print(f"✅ Tersimpan: {save_path}/")

# # ── Simpan encoder ────────────────────────────────────────────────
# for enc_name, encoder in [
#     ("encoder_season",  fit_season),
#     ("encoder_airline", fit_airline),
#     ("encoder_origin",  fit_origin),
#     ("encoder_dest",    fit_dest),
# ]:
#     save_path = f"../models/{enc_name}"
#     if os.path.exists(save_path):
#         shutil.rmtree(save_path)
#     encoder.save(save_path)
#     print(f"✅ Tersimpan: {save_path}/")

# # Simpan FEATURE_COLS dan medians sebagai JSON
# import json
# with open("../models/feature_cols.json", "w") as f:
#     json.dump(FEATURE_COLS, f)
# with open("../models/medians.json", "w") as f:
#     json.dump(medians, f)

# print("\n🎉 Semua artifact berhasil disimpan!")
# print("\n# Cara load kembali (contoh untuk Versi A, GBT Classifier):")
# print("# from pyspark.ml.classification import LogisticRegressionModel, RandomForestClassificationModel, GBTClassificationModel")
# print("# from pyspark.ml.feature import StringIndexerModel")
# print("# gbt_model_A = GBTClassificationModel.load('../models/A_Weighting__gbt_classifier')")
# print("# gbt_model_B = GBTClassificationModel.load('../models/B_Hybrid_Undersample__gbt_classifier')")
# print("# fit_origin  = StringIndexerModel.load('../models/encoder_origin')")
